In [37]:
%pip install numpy pandas scikit-learn kagglehub

Note: you may need to restart the kernel to use updated packages.


# Nienadzorowana detekcja anomalii w strumieniu danych

Ten notebook przygotowuje potok do nienadzorowanej, przyrostowej detekcji anomalii w danych strumieniowych.

Założenia:
- dane spływają partiami,
- model początkowy jest tworzony na pierwszym oknie danych,
- kolejne partie są najpierw punktowane, a potem używane do aktualizacji stanu,
- można stosować dowolną miarę niepodobieństwa do sąsiadów,
- porównanie jest robione z referencyjnymi algorytmami dostępnymi w Pythonie.

Etykieta, jeśli istnieje, jest używana wyłącznie do oceny jakości, a nie do uczenia.

## Założenia wejściowe

Notebook zakłada źródło danych tabelarycznych w pliku CSV, Parquet, Feather, Pickle albo JSON. Potok działa również wtedy, gdy zbiór nie ma etykiety anomalii.

Jeśli etykieta istnieje, notebook może ją automatycznie wykryć i wykorzystać tylko do walidacji porównawczej.

In [38]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import OneClassSVM

DATA_PATH = Path(r"data/raw/creditcard.csv")
BATCH_SIZE = 1000
INITIAL_WINDOW_SIZE = 10000
REFERENCE_WINDOW_SIZE = 20000
N_NEIGHBORS = 5
DISTANCE_METRIC: str | Callable = "euclidean"
ANOMALY_QUANTILE = 0.995
RANDOM_STATE = 42
LABEL_CANDIDATES = ("Class", "class", "label", "Label", "target", "Target", "is_anomaly", "anomaly")

In [39]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Folder gotowy: {RAW_DIR.resolve()}")

Folder gotowy: C:\Users\mati\Desktop\ZUM\data\raw


In [40]:
def load_dataset(data_source: Path | str | pd.DataFrame) -> pd.DataFrame:
    if isinstance(data_source, pd.DataFrame):
        return data_source.copy()

    data_path = Path(data_source)
    if not data_path.exists():
        raise FileNotFoundError(f"Nie znaleziono pliku: {data_path}")

    suffix = data_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(data_path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(data_path)
    if suffix in {".feather", ".ft"}:
        return pd.read_feather(data_path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(data_path)
    if suffix == ".json":
        return pd.read_json(data_path)

    raise ValueError(f"Nieobsługiwany format pliku: {suffix}")


def basic_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = [str(column).strip() for column in cleaned.columns]
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)
    return cleaned


def detect_label_column(df: pd.DataFrame, preferred: str | None = None) -> str | None:
    if preferred and preferred in df.columns:
        return preferred

    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate

    return None


def split_features_and_label(
    df: pd.DataFrame,
    label_column: str | None = None,
) -> tuple[pd.DataFrame, pd.Series | None, str | None]:
    detected_label = detect_label_column(df, preferred=label_column)
    if detected_label is None:
        return df.copy(), None, None

    features = df.drop(columns=[detected_label])
    label = df[detected_label].copy()
    return features, label, detected_label


def infer_feature_groups(features: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_features = features.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [column for column in features.columns if column not in numeric_features]
    return numeric_features, categorical_features

In [41]:
def build_preprocessor(numeric_features: Iterable[str], categorical_features: Iterable[str]) -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, list(numeric_features)),
            ("categorical", categorical_pipeline, list(categorical_features)),
        ],
        remainder="drop",
    )


def to_dense(matrix) -> np.ndarray:
    if hasattr(matrix, "toarray"):
        return matrix.toarray()
    return np.asarray(matrix)


@dataclass
class StreamingNeighborAnomalyDetector:
    n_neighbors: int = N_NEIGHBORS
    metric: str | Callable = DISTANCE_METRIC
    history_size: int = REFERENCE_WINDOW_SIZE
    score_quantile: float = ANOMALY_QUANTILE
    preprocessor: ColumnTransformer | None = None
    numeric_features: list[str] = field(default_factory=list)
    categorical_features: list[str] = field(default_factory=list)
    reference_buffer: deque = field(default_factory=deque)
    threshold_: float = 0.0
    fitted_: bool = False

    def _reference_matrix(self) -> np.ndarray:
        if not self.reference_buffer:
            return np.empty((0, 0), dtype=float)
        return np.asarray(list(self.reference_buffer), dtype=float)

    def _score_components(
        self,
        matrix: np.ndarray,
        reference_matrix: np.ndarray | None = None,
        include_self: bool = False,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        if reference_matrix is None:
            reference_matrix = self._reference_matrix()

        if reference_matrix.size == 0:
            raise ValueError("Detector nie ma jeszcze bazy referencyjnej.")

        n_neighbors = min(self.n_neighbors + int(include_self), len(reference_matrix))
        n_neighbors = max(n_neighbors, 1)
        neighbor_model = NearestNeighbors(n_neighbors=n_neighbors, metric=self.metric)
        neighbor_model.fit(reference_matrix)
        distances, _ = neighbor_model.kneighbors(matrix)

        if include_self and distances.shape[1] > 1:
            distances = distances[:, 1:]

        global_score = distances.mean(axis=1)
        local_scale = distances.std(axis=1) + 1e-9
        local_score = global_score / local_scale
        anomaly_score = 0.6 * global_score + 0.4 * local_score
        return global_score, local_score, anomaly_score

    def _update_threshold(self) -> None:
        reference_matrix = self._reference_matrix()
        if len(reference_matrix) < max(2, self.n_neighbors):
            self.threshold_ = 0.0
            return

        _, _, anomaly_score = self._score_components(reference_matrix, reference_matrix=reference_matrix, include_self=True)
        self.threshold_ = float(np.quantile(anomaly_score, self.score_quantile))

    def fit_initial(self, batch: pd.DataFrame, label_column: str | None = None) -> pd.DataFrame:
        features, labels, detected_label = split_features_and_label(batch, label_column)
        self.numeric_features, self.categorical_features = infer_feature_groups(features)
        self.preprocessor = build_preprocessor(self.numeric_features, self.categorical_features)

        transformed = to_dense(self.preprocessor.fit_transform(features))
        self.reference_buffer = deque((row.astype(float, copy=False) for row in transformed), maxlen=self.history_size)
        self._update_threshold()
        self.fitted_ = True

        global_score, local_score, anomaly_score = self._score_components(
            transformed,
            reference_matrix=self._reference_matrix(),
            include_self=True,
        )
        result = pd.DataFrame(
            {
                "global_score": global_score,
                "local_score": local_score,
                "anomaly_score": anomaly_score,
                "is_anomaly": (anomaly_score > self.threshold_).astype(int),
            },
            index=features.index,
        )
        if labels is not None:
            result["label"] = labels.to_numpy()
        result["detected_label"] = detected_label
        return result

    def score_batch(self, batch: pd.DataFrame, label_column: str | None = None) -> pd.DataFrame:
        if not self.fitted_ or self.preprocessor is None:
            raise ValueError("Detector nie został jeszcze zainicjalizowany.")

        features, labels, detected_label = split_features_and_label(batch, label_column)
        transformed = to_dense(self.preprocessor.transform(features))
        global_score, local_score, anomaly_score = self._score_components(transformed)
        predictions = (anomaly_score > self.threshold_).astype(int)

        result = pd.DataFrame(
            {
                "global_score": global_score,
                "local_score": local_score,
                "anomaly_score": anomaly_score,
                "is_anomaly": predictions,
            },
            index=features.index,
        )
        if labels is not None:
            result["label"] = labels.to_numpy()
        result["detected_label"] = detected_label
        return result

    def update_reference(self, batch: pd.DataFrame, label_column: str | None = None) -> None:
        if not self.fitted_ or self.preprocessor is None:
            raise ValueError("Detector nie został jeszcze zainicjalizowany.")

        features, _, _ = split_features_and_label(batch, label_column)
        transformed = to_dense(self.preprocessor.transform(features))
        for row in transformed:
            self.reference_buffer.append(row.astype(float, copy=False))
        self._update_threshold()

    def process_batch(self, batch: pd.DataFrame, label_column: str | None = None) -> pd.DataFrame:
        scored = self.score_batch(batch, label_column=label_column)
        self.update_reference(batch, label_column=label_column)
        return scored

In [42]:
df_raw = load_dataset(DATA_PATH)
label_column = detect_label_column(df_raw)
stream_results, stream_detector = run_stream_detector(
    df_raw,
    label_column=label_column,
    batch_size=BATCH_SIZE,
    max_batches=3,
    metric=DISTANCE_METRIC,
)

print("Wymiary danych:", df_raw.shape)
print("Wykryta etykieta do oceny:", label_column)
print("Liczba rekordów oznaczonych jako anomalie:", int(stream_results["is_anomaly"].sum()))
print("Próg decyzyjny:", stream_detector.threshold_)
stream_results.head()

Wymiary danych: (284807, 31)
Wykryta etykieta do oceny: Class
Liczba rekordów oznaczonych jako anomalie: 48
Próg decyzyjny: 43.82559004885475


,batch_index,method,global_score,local_score,anomaly_score,is_anomaly,label,detected_label
0,0,neighbor_stream,3.077904,19.468482,9.634135,0,0,Class
1,0,neighbor_stream,1.658866,4.028498,2.606719,0,0,Class
2,0,neighbor_stream,5.261485,7.647615,6.215937,0,0,Class
3,0,neighbor_stream,4.737060,51.274788,23.352151,0,0,Class
4,0,neighbor_stream,3.405051,12.665444,7.109208,0,0,Class


In [43]:
def iter_stream_batches(data_source: Path | str | pd.DataFrame, batch_size: int = BATCH_SIZE) -> Iterable[pd.DataFrame]:
    if isinstance(data_source, pd.DataFrame):
        for start in range(0, len(data_source), batch_size):
            yield data_source.iloc[start:start + batch_size].copy()
        return

    data_path = Path(data_source)
    suffix = data_path.suffix.lower()

    if suffix == ".csv":
        yield from pd.read_csv(data_path, chunksize=batch_size)
        return

    dataset = load_dataset(data_path)
    for start in range(0, len(dataset), batch_size):
        yield dataset.iloc[start:start + batch_size].copy()


def run_stream_detector(
    data_source: Path | str | pd.DataFrame,
    label_column: str | None = None,
    batch_size: int = BATCH_SIZE,
    max_batches: int | None = None,
    metric: str | Callable = DISTANCE_METRIC,
) -> tuple[pd.DataFrame, StreamingNeighborAnomalyDetector]:
    batch_iterator = iter_stream_batches(data_source, batch_size=batch_size)
    first_batch = next(batch_iterator, None)
    if first_batch is None:
        raise ValueError("Brak danych do przetworzenia.")

    detector = StreamingNeighborAnomalyDetector(metric=metric)
    first_result = detector.fit_initial(first_batch, label_column=label_column)
    first_result.insert(0, "batch_index", 0)
    first_result.insert(1, "method", "neighbor_stream")

    results = [first_result]

    for batch_index, batch in enumerate(batch_iterator, start=1):
        if max_batches is not None and batch_index >= max_batches:
            break
        batch_result = detector.process_batch(batch, label_column=label_column)
        batch_result.insert(0, "batch_index", batch_index)
        batch_result.insert(1, "method", "neighbor_stream")
        results.append(batch_result)

    return pd.concat(results, ignore_index=True), detector

## Metody referencyjne

W tej sekcji są wyłącznie algorytmy porównawcze do nienadzorowanej detekcji anomalii: `IsolationForest` i `One-Class SVM`.

Nie używam tu klasycznych modeli nadzorowanych, bo nie pasują do założeń zadania.

In [44]:
def safe_anomaly_metrics(y_true: np.ndarray | None, scores: np.ndarray, flags: np.ndarray) -> dict[str, float]:
    metrics = {
        "roc_auc": np.nan,
        "average_precision": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
    }

    if y_true is None:
        return metrics

    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return metrics

    metrics["roc_auc"] = roc_auc_score(y_true, scores)
    metrics["average_precision"] = average_precision_score(y_true, scores)
    metrics["precision"] = precision_score(y_true, flags, zero_division=0)
    metrics["recall"] = recall_score(y_true, flags, zero_division=0)
    metrics["f1"] = f1_score(y_true, flags, zero_division=0)
    return metrics


@dataclass
class SlidingWindowReferenceModel:
    name: str
    estimator: object
    history_size: int = REFERENCE_WINDOW_SIZE
    score_quantile: float = ANOMALY_QUANTILE
    buffer: deque = field(default_factory=deque)
    threshold_: float = 0.0
    fitted_: bool = False

    def _score_samples(self, matrix: np.ndarray) -> np.ndarray:
        if hasattr(self.estimator, "score_samples"):
            return -np.asarray(self.estimator.score_samples(matrix))
        if hasattr(self.estimator, "decision_function"):
            return -np.asarray(self.estimator.decision_function(matrix))
        raise AttributeError(f"Model {self.name} nie udostepnia score_samples ani decision_function.")

    def _reference_matrix(self) -> np.ndarray:
        if not self.buffer:
            return np.empty((0, 0), dtype=float)
        return np.asarray(list(self.buffer), dtype=float)

    def _append(self, matrix: np.ndarray) -> None:
        for row in matrix:
            self.buffer.append(np.asarray(row, dtype=float))

    def _refit(self) -> None:
        reference_matrix = self._reference_matrix()
        if len(reference_matrix) < 2:
            self.fitted_ = False
            self.threshold_ = 0.0
            return

        self.estimator.fit(reference_matrix)
        self.fitted_ = True
        self.threshold_ = float(np.quantile(self._score_samples(reference_matrix), self.score_quantile))

    def initialize(self, matrix: np.ndarray) -> None:
        self.buffer = deque(maxlen=self.history_size)
        self._append(matrix)
        self._refit()

    def process_matrix(self, matrix: np.ndarray) -> pd.DataFrame:
        if not self.fitted_:
            self.initialize(matrix)

        scores = self._score_samples(matrix)
        flags = (scores > self.threshold_).astype(int)
        result = pd.DataFrame(
            {
                "anomaly_score": scores,
                "is_anomaly": flags,
            }
        )

        self._append(matrix)
        self._refit()
        return result


def benchmark_reference_models(
    data_source: Path | str | pd.DataFrame,
    label_column: str | None = None,
    batch_size: int = BATCH_SIZE,
    max_batches: int | None = None,
) -> pd.DataFrame:
    batch_iterator = iter_stream_batches(data_source, batch_size=batch_size)
    first_batch = next(batch_iterator, None)
    if first_batch is None:
        raise ValueError("Brak danych do benchmarku.")

    features, _, _ = split_features_and_label(first_batch, label_column)
    numeric_features, categorical_features = infer_feature_groups(features)
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    first_matrix = to_dense(preprocessor.fit_transform(features))

    models = {
        "IsolationForest": SlidingWindowReferenceModel(
            name="IsolationForest",
            estimator=IsolationForest(
                n_estimators=100,
                contamination="auto",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
        "OneClassSVM": SlidingWindowReferenceModel(
            name="OneClassSVM",
            estimator=OneClassSVM(
                kernel="rbf",
                nu=0.05,
                gamma="scale",
            ),
        ),
    }

    for model in models.values():
        model.initialize(first_matrix)

    collected: dict[str, dict[str, list[np.ndarray]]] = {
        name: {"scores": [], "flags": [], "labels": []} for name in models
    }

    for batch_index, batch in enumerate(batch_iterator, start=1):
        if max_batches is not None and batch_index >= max_batches:
            break

        features, labels, _ = split_features_and_label(batch, label_column)
        matrix = to_dense(preprocessor.transform(features))

        for name, model in models.items():
            batch_result = model.process_matrix(matrix)
            collected[name]["scores"].append(batch_result["anomaly_score"].to_numpy())
            collected[name]["flags"].append(batch_result["is_anomaly"].to_numpy())
            if labels is not None:
                collected[name]["labels"].append(labels.to_numpy())

    rows: list[dict[str, float | str | int]] = []
    for name, data in collected.items():
        scores = np.concatenate(data["scores"]) if data["scores"] else np.array([])
        flags = np.concatenate(data["flags"]) if data["flags"] else np.array([])
        y_true = np.concatenate(data["labels"]) if data["labels"] else None

        row: dict[str, float | str | int] = {
            "model": name,
            "samples": int(len(scores)),
            "mean_score": float(np.mean(scores)) if len(scores) else np.nan,
            "anomaly_rate": float(np.mean(flags)) if len(flags) else np.nan,
        }
        row.update(safe_anomaly_metrics(y_true, scores, flags))
        rows.append(row)

    summary = pd.DataFrame(rows)
    metric_order = ["roc_auc", "average_precision", "f1", "precision", "recall", "anomaly_rate"]
    sort_column = next((column for column in metric_order if column in summary.columns), "mean_score")
    return summary.sort_values(by=sort_column, ascending=False, na_position="last").reset_index(drop=True)


reference_results = benchmark_reference_models(
    df_raw,
    label_column=label_column,
    batch_size=BATCH_SIZE,
    max_batches=3,
)

reference_results

,model,samples,mean_score,anomaly_rate,roc_auc,average_precision,precision,recall,f1
0,IsolationForest,2000,0.422644,0.0100,NaN,NaN,NaN,NaN,NaN
1,OneClassSVM,2000,-1.643987,0.2945,NaN,NaN,NaN,NaN,NaN
